# BBL 2026 Final — Post-Match Analysis
**Sydney Sixers vs Sydney Thunder**  
This notebook contains the delivery-level analysis underpinning the three insights in the post-match report:
1. Smith vs Right-Arm Pace vs Tanveer Sangha
2. Ryan Hadley's length problem
3. Babar Azam — why Thunder never contained him

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.1f}'.format)

In [ ]:
# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv('/Users/kushgirap/Desktop/bbl-2026-analysis/data/bbl_2026_final_coded - Sheet3.csv')
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
df = df.rename(columns={'ball_speed_(kmph)': 'ball_speed', 'batter_type': 'handedness'})

df['extras']     = df['extras'].fillna(0)
df['ball_speed'] = pd.to_numeric(df['ball_speed'], errors='coerce')
df['striker']    = df['striker'].str.strip()
df['handedness'] = df['handedness'].str.strip()

# Boolean helpers
df['is_wicket']   = df['wicket'] == 'Yes'
df['is_wide']     = df['wide']   == 'Yes'
df['is_dot']      = (df['runs_batter'] == 0) & ~df['is_wide']
df['is_boundary'] = df['runs_batter'].isin([4, 6])
df['is_four']     = df['runs_batter'] == 4
df['is_six']      = df['runs_batter'] == 6
df['is_legal']    = ~df['is_wide']
df['runs_total']  = df['runs_batter'] + df['extras']

# Phase
def assign_phase(over):
    if over <= 5:    return 'Powerplay (1-6)'
    elif over <= 14: return 'Middle (7-15)'
    else:            return 'Death (16-20)'
df['phase'] = df['over'].apply(assign_phase)

# Team mapping
THUNDER = ['Matthew Gilkes', 'David Warner', 'Sam Konstas', 'Sam Billings',
           'Nic Maddinson', 'Chris Green', 'Daniel Sams', "Aidan O'Connor",
           'Tanveer Sangha', 'Ryan Hadley', 'Wes Agar']
SIXERS  = ['Babar Azam', 'Steve Smith', 'Josh Phillipe', 'Moises Henriques',
           'Sam Curran', 'Lachlan Shaw', 'Jack Edwards', 'Sean Abbott',
           'Mitchell Starc', 'Ben Manenti', 'Joel Davis']

df['batting_team'] = df['striker'].apply(
    lambda x: 'Thunder' if x in THUNDER else 'Sixers' if x in SIXERS else 'Unknown'
)
df['bowling_team'] = df['bowler'].apply(
    lambda x: 'Thunder' if x in THUNDER else 'Sixers' if x in SIXERS else 'Unknown'
)

print(f'Total deliveries: {len(df)}')
print(f'Legal deliveries: {df["is_legal"].sum()}')
print(f'Wickets: {df["is_wicket"].sum()}')
print(f'Boundaries: {df["is_boundary"].sum()}')

## Match Overview

In [ ]:
# Innings summary
runs_summary = df.groupby(['innings', 'batting_team']).agg(
    runs       = ('runs_batter', 'sum'),
    extras     = ('extras', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    fours      = ('is_four', 'sum'),
    sixes      = ('is_six', 'sum'),
    wickets    = ('is_wicket', 'sum'),
)
runs_summary['total_runs'] = runs_summary['runs'] + runs_summary['extras']

balls_summary = df[df['is_legal']].groupby(['innings', 'batting_team']).agg(
    balls = ('ball', 'count'),
    dots  = ('is_dot', 'sum'),
)

innings_summary = runs_summary.join(balls_summary)
innings_summary['run_rate'] = (innings_summary['total_runs'] / (innings_summary['balls'] / 6)).round(2)
innings_summary['dot_pct']  = (innings_summary['dots'] / innings_summary['balls'] * 100).round(1)
print(innings_summary)

In [ ]:
# Individual batter scorecard
batter_runs = df.groupby(['innings', 'batting_team', 'striker']).agg(
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    fours      = ('is_four', 'sum'),
    sixes      = ('is_six', 'sum'),
    wickets    = ('is_wicket', 'sum'),
)
batter_balls = df[df['is_legal']].groupby(['innings', 'batting_team', 'striker']).agg(
    balls = ('ball', 'count'),
    dots  = ('is_dot', 'sum'),
)
batter_card = batter_runs.join(batter_balls).reset_index()
batter_card['strike_rate'] = (batter_card['runs'] / batter_card['balls'] * 100).round(1)
batter_card['dot_pct']     = (batter_card['dots'] / batter_card['balls'] * 100).round(1)
print(batter_card.sort_values(['innings', 'runs'], ascending=[True, False]).to_string())

In [ ]:
# Bowler summary
bowler_runs = df.groupby(['innings', 'bowling_team', 'bowler']).agg(
    runs       = ('runs_batter', 'sum'),
    extras     = ('extras', 'sum'),
    wickets    = ('is_wicket', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    beat_bat   = ('beaten', lambda x: (x == 'Yes').sum()),
)
bowler_runs['total_runs'] = bowler_runs['runs'] + bowler_runs['extras']
bowler_balls = df[df['is_legal']].groupby(['innings', 'bowling_team', 'bowler']).agg(
    balls = ('ball', 'count'),
    dots  = ('is_dot', 'sum'),
)
bowler_summary = bowler_runs.join(bowler_balls).reset_index()
bowler_summary['economy'] = (bowler_summary['total_runs'] / (bowler_summary['balls'] / 6)).round(2)
bowler_summary['dot_pct'] = (bowler_summary['dots'] / bowler_summary['balls'] * 100).round(1)
print(bowler_summary.sort_values(['innings', 'economy']).to_string())

---
## Insight 1 — Smith vs Right-Arm Pace vs Tanveer Sangha
Smith was unplayable against pace (SR 341, 7 sixes) but Sangha's leg spin held him to SR 100 in 12 balls.
The blueprint existed — it was deployed too late.

In [ ]:
smith = df[df['striker'] == 'Steve Smith'].copy()
smith = smith.sort_values(['innings', 'over', 'ball']).reset_index(drop=True)

print('Smith — vs each bowler (SR, dots, boundaries):')
print(smith.groupby(['bowler', 'type_of_bowler']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    dots       = ('is_dot', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    beat_bat   = ('beaten', lambda x: (x == 'Yes').sum()),
).assign(sr=lambda x: (x['runs'] / x['balls'] * 100).round(1)).to_string())

In [ ]:
print('Smith vs Sangha — line/length breakdown:')
print(smith[smith['bowler'] == 'Tanveer Sangha'].groupby(['line', 'length']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    dots       = ('is_dot', 'sum'),
    boundaries = ('is_boundary', 'sum'),
).to_string())

In [ ]:
print('Smith vs pace — line/length and shot selection:')
pace_types = ['Right Medium Fast', 'Right Fast']
print(smith[smith['type_of_bowler'].isin(pace_types)].groupby(['line', 'length', 'shot_played']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
).sort_values('runs', ascending=False).to_string())

---
## Insight 2 — Ryan Hadley's Length Problem
Hadley bowled good length outside off to Smith 8 times without adjusting — 26 runs conceded from one zone.

In [ ]:
hadley = df[df['bowler'] == 'Ryan Hadley'].copy()

print('Hadley — who did he bowl to:')
print(hadley.groupby('striker').agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    wickets    = ('is_wicket', 'sum'),
    dots       = ('is_dot', 'sum'),
).sort_values('runs', ascending=False).to_string())

In [ ]:
print('Hadley — line/length breakdown:')
print(hadley.groupby(['line', 'length']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    dots       = ('is_dot', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    wickets    = ('is_wicket', 'sum'),
).assign(sr=lambda x: (x['runs'] / x['balls'] * 100).round(1)).sort_values('runs', ascending=False).to_string())

In [ ]:
print('Hadley — shot by shot (top 10 by runs):')
print(hadley.groupby(['line', 'length', 'shot_played']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
).sort_values('runs', ascending=False).head(10).to_string())

print('\nHadley — by phase:')
print(hadley.groupby('phase').agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    wickets    = ('is_wicket', 'sum'),
).assign(economy=lambda x: (x['runs'] / x['balls'] * 6).round(2)).to_string())

---
## Insight 3 — Babar Azam: Thunder Never Found a Plan
Babar scored 47 off 39 largely unopposed. Green held him to SR 80 with a tight middle-stump line.
No other Thunder bowler replicated it.

In [ ]:
babar = df[df['striker'] == 'Babar Azam'].copy()

print('Babar — overall:')
print(babar.agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    wickets    = ('is_wicket', 'sum'),
    dots       = ('is_dot', 'sum'),
))

In [ ]:
print('Babar — who bowled to him:')
print(babar.groupby(['bowler', 'type_of_bowler']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    dots       = ('is_dot', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    wickets    = ('is_wicket', 'sum'),
).assign(sr=lambda x: (x['runs'] / x['balls'] * 100).round(1)).sort_values('runs', ascending=False).to_string())

In [ ]:
print('Babar — line/length breakdown:')
print(babar.groupby(['line', 'length']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    dots       = ('is_dot', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    wickets    = ('is_wicket', 'sum'),
).assign(sr=lambda x: (x['runs'] / x['balls'] * 100).round(1)).sort_values('runs', ascending=False).to_string())

In [ ]:
print('Babar — shot selection (top 15 by runs):')
print(babar.groupby(['line', 'length', 'shot_played']).agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
).sort_values('runs', ascending=False).head(15).to_string())

print('\nBabar — by phase:')
print(babar.groupby('phase').agg(
    balls      = ('is_legal', 'sum'),
    runs       = ('runs_batter', 'sum'),
    boundaries = ('is_boundary', 'sum'),
    dots       = ('is_dot', 'sum'),
).assign(sr=lambda x: (x['runs'] / x['balls'] * 100).round(1)).to_string())

print('\nBabar — dismissal detail:')
print(babar[babar['is_wicket'] == 1][[
    'bowler', 'type_of_bowler', 'line', 'length',
    'shot_played', 'dismissal_type', 'edged', 'beaten', 'ball_speed'
]])